# Ablation Evaluation — NLG + Grounding metrics per checkpoint

Loops over every checkpoint in `ablation_study/weights/<config>/`, rebuilds the
projector and LoRA-LLM with the CORRECT `num_queries` / `lora_r` for that config
(parsed from the folder name), generates reports, and computes:
- **NLG:** BLEU-1..4, ROUGE-L, METEOR, BERTScore
- **Grounding:** mean heatmap pairwise similarity (collapse check)

Then assembles one final ablation table.

**Note on Pointing Game / IoU@0.5:** those require the NIH ChestX-ray8 annotated
subset and your bounding-box localization code. This notebook computes NLG +
heatmap-variance for every config automatically; a placeholder cell shows where
to plug in your existing NIH localization function if you want PG/IoU per config.

## Cell 1 — Install + mount

In [ ]:
!pip install -q -U bitsandbytes transformers peft accelerate datasets evaluate rouge_score nltk bert_score
from google.colab import drive
drive.mount('/content/drive')
import nltk
nltk.download('punkt'); nltk.download('punkt_tab'); nltk.download('wordnet')

## Cell 2 — Imports, device, paths

In [ ]:
import os, re, gc, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm
from datasets import load_dataset
from transformers import (AutoImageProcessor, AutoModel, AutoTokenizer,
                          AutoModelForCausalLM, BitsAndBytesConfig)
from peft import PeftModel, LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

ABLATION_ROOT = "/content/drive/MyDrive/ablation_study"
WEIGHTS_ROOT  = os.path.join(ABLATION_ROOT, "weights")
EVAL_CSV      = os.path.join(ABLATION_ROOT, "ablation_eval_metrics.csv")
SEED = 42
N_SAMPLE_HEATMAP = 150   # images used for the heatmap-variance collapse check

## Cell 3 — Load base models ONCE (vision encoder, tokenizer, base LLM)

In [ ]:
print("Loading Rad-DINO + tokenizer...")
vision_processor = AutoImageProcessor.from_pretrained("microsoft/rad-dino")
vision_encoder = AutoModel.from_pretrained("microsoft/rad-dino").to(device)
vision_encoder.eval()

LLM_ID = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                         bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
print("Loading base Qwen (4-bit) once...")
base_llm = AutoModelForCausalLM.from_pretrained(LLM_ID, quantization_config=bnb,
                                                device_map={"": 0})
LLM_HIDDEN = base_llm.config.hidden_size
print("Base models ready. LLM hidden size:", LLM_HIDDEN)

## Cell 4 — Holdout test set (same split/seed as training)

In [ ]:
print("Loading test split...")
full = load_dataset("Shrey-1329/cxiu_hf_dataset", split="train")
tt = full.train_test_split(test_size=0.2, seed=SEED)
vt = tt['test'].train_test_split(test_size=0.5, seed=SEED)
test_split = vt['test']   # untouched 10%
print("Test images:", len(test_split))

## Cell 5 — Projector definition (with return_attention for grounding)

In [ ]:
class QFormerProjector(nn.Module):
    def __init__(self, num_queries, llm_dim, vision_dim=768):
        super().__init__()
        self.query_tokens = nn.Parameter(torch.randn(1, num_queries, vision_dim))
        self.cross_attn = nn.MultiheadAttention(vision_dim, num_heads=8, batch_first=True)
        self.norm1 = nn.LayerNorm(vision_dim)
        self.ffn = nn.Sequential(nn.Linear(vision_dim, vision_dim*4), nn.GELU(),
                                 nn.Linear(vision_dim*4, vision_dim))
        self.norm2 = nn.LayerNorm(vision_dim)
        self.proj = nn.Linear(vision_dim, llm_dim)
        self.llm_norm = nn.LayerNorm(llm_dim)
    def forward(self, patch_embeddings, return_attention=False):
        sp = patch_embeddings[:, 1:, :]
        q = self.query_tokens.expand(sp.shape[0], -1, -1)
        a, attn = self.cross_attn(q, sp, sp, need_weights=return_attention,
                                  average_attn_weights=True)
        x = self.norm1(q + a)
        x = self.norm2(x + self.ffn(x))
        out = self.llm_norm(self.proj(x))
        if return_attention:
            return out, attn
        return out

## Cell 6 — Helpers: parse config name, load a checkpoint

Folder names look like `K32_r16`. We parse `num_queries` and `lora_r` from the
name so the projector and LoRA adapters are rebuilt at the CORRECT sizes —
this is what prevents the `load_state_dict` shape-mismatch.

In [ ]:
def parse_config_name(name):
    """K32_r16 -> (32, 16)"""
    k = int(re.search(r"K(\d+)", name).group(1))
    r = int(re.search(r"r(\d+)", name).group(1))
    return k, r

def load_checkpoint(config_name):
    k, r = parse_config_name(config_name)
    ckpt_dir = os.path.join(WEIGHTS_ROOT, config_name)

    # rebuild projector at correct K
    projector = QFormerProjector(num_queries=k, llm_dim=LLM_HIDDEN).to(device)
    projector.load_state_dict(
        torch.load(os.path.join(ckpt_dir, "vision_projector.pth"),
                   map_location=device, weights_only=True))
    projector.eval()

    # attach the LoRA adapters for this config to the shared base LLM
    llm = PeftModel.from_pretrained(base_llm,
            os.path.join(ckpt_dir, "qwen_lora_adapters"))
    llm.eval()
    return projector, llm, k, r

## Cell 7 — Generation for one checkpoint

In [ ]:
EVAL_PROMPT = ("<|im_start|>system\n"
    "You are an expert radiologist. Write ONLY the diagnostic narrative.<|im_end|>\n"
    "<|im_start|>user\nAnalyze this chest radiograph and generate a concise clinical report.<|im_end|>\n"
    "<|im_start|>assistant\n")

def generate_reports(projector, llm):
    prompt_ids = tokenizer(EVAL_PROMPT, add_special_tokens=False,
                           return_tensors="pt").input_ids.to(device)
    gens, refs = [], []
    with torch.no_grad():
        for item in tqdm(test_split, desc="generating", leave=False):
            image = item['image'].convert('RGB')
            pv = vision_processor(images=image, return_tensors="pt").pixel_values.to(device)
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                pe = vision_encoder(pv).last_hidden_state.to(torch.float16)
                vprompt = projector(pe)
                pe_txt = llm.get_input_embeddings()(prompt_ids)
                emb = torch.cat([vprompt, pe_txt], dim=1)
                am = torch.ones(emb.shape[:2], dtype=torch.long, device=device)
                out = llm.generate(inputs_embeds=emb, attention_mask=am,
                                   max_new_tokens=120, min_new_tokens=20,
                                   num_beams=4, do_sample=False, length_penalty=1.2,
                                   repetition_penalty=1.1, early_stopping=True,
                                   pad_token_id=tokenizer.pad_token_id,
                                   eos_token_id=tokenizer.eos_token_id)
            gen = tokenizer.decode(out[0], skip_special_tokens=True).strip()
            ref = item.get('text', None)
            if ref is None or not ref.strip():
                continue
            gens.append(gen); refs.append(ref.strip())
    return gens, refs

## Cell 8 — NLG metrics (nltk BLEU 1-4, ROUGE-L, METEOR, BERTScore)

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

_smooth = SmoothingFunction().method1
_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
_bw = {"BLEU-1":(1,0,0,0), "BLEU-2":(0.5,0.5,0,0),
       "BLEU-3":(1/3,1/3,1/3,0), "BLEU-4":(0.25,0.25,0.25,0.25)}

def compute_nlg(gens, refs):
    bl = {k: [] for k in _bw}; rl = []; me = []
    for g, r in zip(gens, refs):
        gt, rt = g.lower().split(), r.lower().split()
        for name, w in _bw.items():
            bl[name].append(sentence_bleu([rt], gt, weights=w, smoothing_function=_smooth))
        rl.append(_rouge.score(r, g)['rougeL'].fmeasure)
        me.append(meteor_score([rt], gt))
    P, R, F1 = bert_score_fn(gens, refs, lang="en", verbose=False)
    out = {k: float(np.mean(v)) for k, v in bl.items()}
    out["ROUGE-L"] = float(np.mean(rl))
    out["METEOR"] = float(np.mean(me))
    out["BERTScore-F1"] = float(F1.mean().item())
    # uniqueness / collapse stat
    out["unique_reports"] = len(set(g.strip() for g in gens))
    out["total_reports"] = len(gens)
    out["pct_unique"] = round(100*out["unique_reports"]/max(out["total_reports"],1), 1)
    return out

## Cell 9 — Heatmap-variance (grounding collapse check) per config

In [ ]:
def compute_heatmap_variance(projector, n_sample=N_SAMPLE_HEATMAP):
    idxs = list(range(min(n_sample, len(test_split))))
    maps = []
    with torch.no_grad():
        for i in tqdm(idxs, desc="heatmaps", leave=False):
            image = test_split[i]['image'].convert('RGB')
            pv = vision_processor(images=image, return_tensors="pt").pixel_values.to(device)
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                pe = vision_encoder(pv).last_hidden_state.to(torch.float16)
                _, attn = projector(pe, return_attention=True)
            vec = attn.squeeze(0).mean(dim=0).float().cpu().numpy()
            lo, hi = vec.min(), vec.max()
            if hi - lo > 1e-8: vec = (vec - lo)/(hi - lo)
            maps.append(vec)
    H = np.stack(maps, 0)
    norms = np.linalg.norm(H, axis=1, keepdims=True); norms[norms==0] = 1e-8
    Hn = H / norms
    rng = np.random.default_rng(SEED); n = Hn.shape[0]
    ii = rng.integers(0, n, 2000); jj = rng.integers(0, n, 2000); m = ii != jj
    cos = np.sum(Hn[ii[m]] * Hn[jj[m]], axis=1)
    return float(cos.mean())

## Cell 10 — (Optional) NIH Pointing Game / IoU per config

Plug your existing NIH localization function here if you want PG/IoU for every
config. It needs the NIH annotated subset + bounding boxes. Left as a stub so
the notebook runs end-to-end without NIH data; returns None if not implemented.

In [ ]:
def compute_nih_localization(projector, llm):
    # TODO: paste your existing NIH Pointing Game / IoU@0.5 code here,
    # using projector(..., return_attention=True) to get the heatmap.
    # Return (pointing_game, iou_at_0p5) or (None, None) if skipping.
    return None, None

## Cell 11 — Run evaluation over ALL checkpoints

Resume-safe: skips configs already in the eval CSV. Frees the PEFT adapters
between configs so VRAM doesn't accumulate.

In [ ]:
config_names = sorted([d for d in os.listdir(WEIGHTS_ROOT)
                       if os.path.isdir(os.path.join(WEIGHTS_ROOT, d))])
print("Found configs:", config_names)

rows = []
if os.path.exists(EVAL_CSV):
    prev = pd.read_csv(EVAL_CSV); rows = prev.to_dict("records")
    done = set(prev["config"]); print("Already evaluated:", done)
else:
    done = set()

for name in config_names:
    if name in done:
        print(f"skip {name}"); continue
    print(f"\n===== EVALUATING {name} =====")
    projector, llm, k, r = load_checkpoint(name)

    gens, refs = generate_reports(projector, llm)
    nlg = compute_nlg(gens, refs)
    hmap_sim = compute_heatmap_variance(projector)
    pg, iou = compute_nih_localization(projector, llm)

    row = {"config": name, "num_queries": k, "lora_r": r,
           **{kk: round(vv, 4) if isinstance(vv, float) else vv
              for kk, vv in nlg.items()},
           "heatmap_pairwise_sim": round(hmap_sim, 4),
           "pointing_game": pg, "iou@0.5": iou}
    rows.append(row)
    pd.DataFrame(rows).to_csv(EVAL_CSV, index=False)
    print(f"  -> logged {name}")

    # free the per-config adapters + projector
    del projector, llm
    gc.collect(); torch.cuda.empty_cache()

print("\nDONE.")
final = pd.DataFrame(rows)
print(final)

## Cell 12 — Assemble the final ablation table

In [ ]:
df = pd.read_csv(EVAL_CSV)
# order rows: K-ablation first (r=16), then rank-ablation (K=64)
df = df.sort_values(["lora_r", "num_queries"]).reset_index(drop=True)

cols = ["config","num_queries","lora_r","METEOR","BERTScore-F1",
        "ROUGE-L","BLEU-4","pct_unique","heatmap_pairwise_sim"]
cols = [c for c in cols if c in df.columns]
table = df[cols]
print("=== FINAL ABLATION TABLE ===")
print(table.to_string(index=False))

table.to_csv(os.path.join(ABLATION_ROOT, "final_ablation_table.csv"), index=False)
print("\nSaved final_ablation_table.csv")